# Bag Of Words And TF-IDF

# Import Required Libraries

In [1]:
import math
from collections import Counter

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Create Text Corpus

In [2]:
corpus = [
    "payment failed payment pending",
    "login failed reset password",
    "payment refund processed",
]

tokens = [
    document.lower().split()
    for document in corpus
]

vocabulary = sorted({
    word
    for documentTokens in tokens
    for word in documentTokens
})

print("Documents:", len(corpus))
print("Vocabulary:", vocabulary)

Documents: 3
Vocabulary: ['failed', 'login', 'password', 'payment', 'pending', 'processed', 'refund', 'reset']


# Build Manual Bag

In [3]:
manualBow = []

for documentTokens in tokens:
    wordCounts = Counter(documentTokens)

    manualBow.append([
        wordCounts[word]
        for word in vocabulary
    ])

manualBowFrame = pd.DataFrame(
    manualBow,
    columns=vocabulary,
    index=["D1", "D2", "D3"],
)

display(manualBowFrame)

,failed,login,password,payment,pending,processed,refund,reset
D1,1,0,0,2,1,0,0,0
D2,1,1,1,0,0,0,0,1
D3,0,0,0,1,0,1,1,0


# Validate With Scikit

In [4]:
countVectorizer = CountVectorizer(
    vocabulary=vocabulary,
)

countMatrix = countVectorizer.fit_transform(corpus)

scikitBowFrame = pd.DataFrame(
    countMatrix.toarray(),
    columns=countVectorizer.get_feature_names_out(),
    index=["D1", "D2", "D3"],
)

display(scikitBowFrame)

bowMatches = np.array_equal(
    manualBowFrame.to_numpy(),
    scikitBowFrame.to_numpy(),
)

print("Manual and Scikit-learn BoW match:", bowMatches)

,failed,login,password,payment,pending,processed,refund,reset
D1,1,0,0,2,1,0,0,0
D2,1,1,1,0,0,0,0,1
D3,0,0,0,1,0,1,1,0


Manual and Scikit-learn BoW match: True


# Calculate Manual TF-IDF

In [5]:
documentCount = len(corpus)
documentFrequency = {}

for word in vocabulary:
    documentFrequency[word] = sum(
        word in documentTokens
        for documentTokens in tokens
    )

idfValues = {
    word: math.log(
        (1 + documentCount)
        / (1 + documentFrequency[word])
    ) + 1
    for word in vocabulary
}

manualTfidf = []

for documentTokens in tokens:
    wordCounts = Counter(documentTokens)

    row = np.array([
        wordCounts[word] * idfValues[word]
        for word in vocabulary
    ])

    rowNorm = np.linalg.norm(row)

    if rowNorm > 0:
        row = row / rowNorm

    manualTfidf.append(row)

manualTfidfFrame = pd.DataFrame(
    manualTfidf,
    columns=vocabulary,
    index=["D1", "D2", "D3"],
)

display(
    pd.DataFrame({
        "word": vocabulary,
        "document_frequency": [
            documentFrequency[word]
            for word in vocabulary
        ],
        "idf": [
            idfValues[word]
            for word in vocabulary
        ],
    }).round(4)
)

display(manualTfidfFrame.round(4))

,word,document_frequency,idf
0,failed,2,1.2877
1,login,1,1.6931
2,password,1,1.6931
3,payment,2,1.2877
4,pending,1,1.6931
5,processed,1,1.6931
6,refund,1,1.6931
7,reset,1,1.6931


,failed,login,password,payment,pending,processed,refund,reset
D1,0.3855,0.0000,0.0000,0.7710,0.5069,0.0000,0.0000,0.0000
D2,0.4020,0.5286,0.5286,0.0000,0.0000,0.0000,0.0000,0.5286
D3,0.0000,0.0000,0.0000,0.4736,0.0000,0.6228,0.6228,0.0000


# Compare TF-IDF Results

In [6]:
tfidfVectorizer = TfidfVectorizer(
    vocabulary=vocabulary,
)

tfidfMatrix = tfidfVectorizer.fit_transform(corpus)

scikitTfidfFrame = pd.DataFrame(
    tfidfMatrix.toarray(),
    columns=tfidfVectorizer.get_feature_names_out(),
    index=["D1", "D2", "D3"],
)

difference = np.abs(
    manualTfidfFrame.to_numpy()
    - scikitTfidfFrame.to_numpy()
)

display(scikitTfidfFrame.round(4))
print("Maximum difference:", difference.max())
print(
    "Manual and Scikit-learn TF-IDF match:",
    np.allclose(
        manualTfidfFrame.to_numpy(),
        scikitTfidfFrame.to_numpy(),
    ),
)

,failed,login,password,payment,pending,processed,refund,reset
D1,0.3855,0.0000,0.0000,0.7710,0.5069,0.0000,0.0000,0.0000
D2,0.4020,0.5286,0.5286,0.0000,0.0000,0.0000,0.0000,0.5286
D3,0.0000,0.0000,0.0000,0.4736,0.0000,0.6228,0.6228,0.0000


Maximum difference: 0.0
Manual and Scikit-learn TF-IDF match: True
